# Определение перспективного тарифа для телеком-компании

Цели и исследования:

- Определить какой тарифный план приносит наибольший доход для компании

Так же необходимо ответить на следующие вопросы:

- Сколько звонков делает каждый пользователь в месяц
- Сколько тратит минут каждый пользователь в месяц
- Сколько сообщений отправляет каждый пользователь в месяц
- Какой объем интернет трафика использует каждый пользователь в месяц
- Какая выручка с каждого пользователя в месяц
- Определить сколько минут, сообщений и трафика требуется пользователям для каждого тарифа в месяц

Проверить следующие гипотезы:

- Средняя выручка пользователей между тарифами «Ультра» и «Смарт» различаются
- Средняя выручка пользователей в Москве отличаются от средней выручки пользователей из других регионов


## Импорт библиотек и данных

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from datetime import datetime, timedelta

In [2]:
calls = pd.read_csv('data/raw/calls.csv')
internet = pd.read_csv('data/raw/internet.csv')
messages = pd.read_csv('data/raw/messages.csv')
tariffs = pd.read_csv('data/raw/tariffs.csv')
users = pd.read_csv('data/raw/users.csv')

### Описание полей таблиц

**Таблица users**
- `user_id` — уникальный идентификатор пользователя
- `first_name` — имя пользователя
- `last_name` — фамилия пользователя
- `age` — возраст пользователя 
- `reg_date` — дата подключения тарифа (день, месяц, год)
- `churn_date` — дата прекращения пользования тарифом 
- `city` — город проживания пользователя
- `tariff` — название тарифного плана

**Таблица calls**
- `id` — уникальный номер звонка
- `call_date` — дата звонка
- `duration` — длительность звонка в минутах
- `user_id` — идентификатор пользователя, сделавшего звонок

**Таблица messages**
- `id` — уникальный номер сообщения
- `message_date` — дата сообщения
- `user_id` — идентификатор пользователя, отправившего сообщение

**Таблица internet**
- `id` — уникальный номер сессии
- `mb_used` — объём потраченного за сессию интернет-трафика (в мегабайтах)
- `session_date` — дата интернет-сессии
- `user_id` — идентификатор пользователя

**Таблица tariffs**
- `tariff_name` — название тарифа
- `rub_monthly_fee` — ежемесячная абонентская плата в рублях
- `minutes_included` — количество минут разговора в месяц, включённых в абонентскую плату
- `messages_included` — количество сообщений в месяц, включённых в абонентскую плату
- `mb_per_month_included` — объём интернет-трафика, включённого в абонентскую плату (в мегабайтах)
- `rub_per_minute` — стоимость минуты разговора сверх тарифного пакета (например, если в тарифе 100 минут разговора в месяц, то со 101 минуты будет взиматься плата)
- `rub_per_message` — стоимость отправки сообщения сверх тарифного пакета
- `rub_per_gb` — стоимость дополнительного гигабайта интернет-трафика сверх тарифного пакета (1 гигабайт = 1024 мегабайта)

## Знакомство с данными

In [3]:
def get_info(df):
  print(f"Кол-во строк: {len(df)} \nКол-во колонок: {df.shape[1]}\n"
    f"Дубликаты: {df.duplicated().sum()}\n"
    f"Память: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

  display(df.head())
  print('___________')
  
  print(f"{'GENERAL'}")
  profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "nunique": df.nunique(dropna=False),
    "na": df.isna().sum(),
    "na%": (df.isna().mean() * 100).round(2),
  })
  display(profile.T)
  
  print('___________')
  num = df.select_dtypes(include="number")
  if not num.empty:
    print(f"DESCRIBE")
    display(num.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).round(2))

In [4]:
HEADER = "\033[0;37;40m"
RESET = "\033[0m"
tables = [calls, internet, messages, tariffs, users]
names = ['calls', 'internet', 'messages', 'tariffs', 'users']

for table, name in zip(tables, names):
  print(f'{HEADER} ___{name.upper()}___ {RESET}')
  get_info(table)
  print(f'Конец \n')


 ___CALLS___ 
Кол-во строк: 202607 
Кол-во колонок: 4
Дубликаты: 0
Память: 26.7 MB


,id,call_date,duration,user_id
0,1000_0,2018-07-25,0.00,1000
1,1000_1,2018-08-17,0.00,1000
2,1000_2,2018-06-11,2.85,1000
3,1000_3,2018-09-21,13.80,1000
4,1000_4,2018-12-15,5.18,1000


___________
GENERAL


,id,call_date,duration,user_id
dtype,str,str,float64,int64
nunique,202607,365,2871,492
na,0,0,0,0
na%,0.0,0.0,0.0,0.0


___________
DESCRIBE


...

___________
GENERAL


,user_id,age,churn_date,city,first_name,last_name,reg_date,tariff
dtype,int64,int64,str,str,str,str,str,str
nunique,500,58,34,76,246,419,270,2
na,0,0,462,0,0,0,0,0
na%,0.0,0.0,92.4,0.0,0.0,0.0,0.0,0.0


___________
DESCRIBE


,user_id,age
count,500.00,500.00
mean,1249.50,46.59
std,144.48,16.67
min,1000.00,18.00
1%,1004.99,18.00
5%,1024.95,21.00
25%,1124.75,32.00
50%,1249.50,46.00
75%,1374.25,62.00
95%,1474.05,72.05


Конец 



### Краткий вывод по данным

**Таблица `calls`**
- В таблице 202607 записи, дубликатов и пропусков нет. 
- Все IDs уникальные. 
- Колонка `call_date` имеет тип данных `str` в следующем шаге необходимо преобразовать ее в `datetime`
- `duration` до 5го процентиля равен 0 - это недозвоны

**Таблица `internet`**
- В таблице 149396 записей, дубликатов и пропусков нет. 
- `session_date` имеет тип данных `str` в следующем шаге необходимо преобразовать ее в `datetime`
- Лишняя колонка `unnamed` дублирующая индекс

**Таблица `messages`**
- В таблице 123036 записей, дубликатов и пропусков нет. 
- `message_date` имеет тип данных `str` в следующем шаге необходимо преобразовать ее в `datetime`

**Таблица `tariffs`**
- 2 строки с описанием планов
- Нет недостающих значений, нет дубликатов, все данные соответствуют типу данных

**Таблица `users`**
- В таблице 500 записей, дубликатов. 
- `reg_date`, `churn_date` имеют тип данных `str` в следующем шаге необходимо преобразовать ее в `datetime`
- В колонке `churn_date` есть пропущенные значения, свзязанно это с тем, что у пользователя еще действует тариф
